03_Summarization_Gemini.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/github/urcraft/data_analytics_lecture_notebooks/blob/main/03_Summarization_Gemini.ipynb

<a target="_blank" href="https://colab.research.google.com/github/urcraft/data_analytics_lecture_notebooks/blob/main/03_Summarization_Gemini.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Summarization with Gemini: Condensing Business Documents

## What you will learn
- Use an LLM to summarize documents for business audiences.
- Compare LLM summaries against human-written reference summaries.
- Identify summarization failures: omission, hallucination, misleading emphasis.

**NLP Task**: Summarization — condensing longer content into shorter, actionable form.

**Dataset**: `EdinburghNLP/xsum` from HuggingFace — BBC news articles with
single-sentence reference summaries. We'll treat these as "executive briefs."

Expected runtime: 5–10 minutes
Expected cost: Free-tier Gemini


In [ ]:
%pip install -q google-genai datasets pandas


In [ ]:
import os
import json
import time
import pandas as pd

# --- Gemini Setup ---
GEMINI_MODEL = 'gemini-3.1-flash-lite-preview'
print('Using Gemini model:', GEMINI_MODEL)

from google import genai

api_key = os.getenv('GOOGLE_API_KEY')
if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get('GOOGLE_API_KEY')
    except Exception:
        api_key = None

if not api_key:
    raise ValueError('Set GOOGLE_API_KEY environment variable (or Colab secret).')

client = genai.Client(api_key=api_key)
print('Gemini client ready.')


## Load Dataset

XSum contains BBC articles with one-sentence reference summaries.
We take 10 articles to keep demo fast.


In [ ]:
from datasets import load_dataset

ds = load_dataset('EdinburghNLP/xsum', split='test')

# Take 10 varied articles (skip very short ones)
import random
random.seed(42)

candidates = [i for i in range(len(ds)) if len(ds[i]['document']) > 500]
sample_indices = random.sample(candidates, 10)

sample_df = pd.DataFrame([ds[i] for i in sample_indices])
sample_df = sample_df.rename(columns={'document': 'article', 'summary': 'reference_summary'})

print(f'Sample size: {len(sample_df)}')
for i, row in sample_df.head(3).iterrows():
    print(f'\nArticle {i} ({len(row["article"])} chars):')
    print(f'  First 150 chars: {row["article"][:150]}...')
    print(f'  Reference summary: {row["reference_summary"]}')


## Summarize with Gemini

We ask Gemini for a 1-2 sentence executive summary, similar to
"Summarize this report in one sentence for the board."


In [ ]:
SUMMARIZATION_PROMPT = """You are writing an executive brief. Summarize the following article
in exactly 1-2 sentences. Focus on the most important finding or event.
Do not add any information not present in the article.

Article:
\"\"\"{article}\"\"\"

Summary:"""

def summarize_text(article: str) -> dict:
    prompt = SUMMARIZATION_PROMPT.format(article=article)
    try:
        resp = client.models.generate_content(model=GEMINI_MODEL, contents=prompt)
        return {'summary': resp.text.strip(), 'error': None}
    except Exception as e:
        return {'summary': None, 'error': str(e)}

results = []
for idx, row in sample_df.iterrows():
    result = summarize_text(row['article'])
    results.append(result)
    time.sleep(0.5)

sample_df['llm_summary'] = [r['summary'] for r in results]
sample_df['error'] = [r['error'] for r in results]

print('Summarization complete.')


## Evaluate Summaries

We use both simple code-based checks AND an LLM-as-judge approach.
This connects to Day 2's evaluation design framework.

### Code-Based Checks (Deterministic)


In [ ]:
# Check 1: Length — is it actually concise?
sample_df['llm_word_count'] = sample_df['llm_summary'].fillna('').apply(lambda x: len(x.split()))
sample_df['ref_word_count'] = sample_df['reference_summary'].apply(lambda x: len(x.split()))
sample_df['length_ok'] = sample_df['llm_word_count'] <= 60  # 1-2 sentences should be < 60 words

print('=== Code-Based Checks ===')
print(f'Length OK (≤60 words): {sample_df["length_ok"].mean():.0%}')
print(f'Average LLM summary length: {sample_df["llm_word_count"].mean():.0f} words')
print(f'Average reference length: {sample_df["ref_word_count"].mean():.0f} words')


### LLM-as-Judge (Faithfulness Check)

We ask a second LLM call to judge whether the summary is faithful to the source.
This is a key evaluation technique you'll explore more on Day 2.


In [ ]:
JUDGE_PROMPT = """You are evaluating a summary for faithfulness.

Original article:
\"\"\"{article}\"\"\"

Summary to evaluate:
\"\"\"{summary}\"\"\"

Answer these questions with YES or NO, then provide a brief explanation:
1. FAITHFUL: Does the summary only contain information present in the article? (no hallucinations)
2. KEY_POINT: Does the summary capture the most important point of the article?
3. MISLEADING: Does the summary give a misleading impression of the article's content?

Respond in this exact JSON format:
{{"faithful": "YES/NO", "key_point": "YES/NO", "misleading": "YES/NO", "explanation": "brief note"}}"""

def judge_summary(article: str, summary: str) -> dict:
    prompt = JUDGE_PROMPT.format(article=article[:3000], summary=summary)  # truncate long articles
    try:
        resp = client.models.generate_content(model=GEMINI_MODEL, contents=prompt)
        raw = resp.text.strip()
        if raw.startswith('```'):
            raw = raw.split('\n', 1)[1] if '\n' in raw else raw[3:]
        if raw.endswith('```'):
            raw = raw.rsplit('```', 1)[0]
        return json.loads(raw.strip())
    except Exception as e:
        return {'faithful': 'ERROR', 'key_point': 'ERROR', 'misleading': 'ERROR', 'explanation': str(e)}

judge_results = []
for idx, row in sample_df.iterrows():
    if row['llm_summary']:
        result = judge_summary(row['article'], row['llm_summary'])
    else:
        result = {'faithful': 'N/A', 'key_point': 'N/A', 'misleading': 'N/A', 'explanation': 'No summary generated'}
    judge_results.append(result)
    time.sleep(0.5)

sample_df['faithful'] = [r.get('faithful', 'ERROR') for r in judge_results]
sample_df['key_point'] = [r.get('key_point', 'ERROR') for r in judge_results]
sample_df['misleading'] = [r.get('misleading', 'ERROR') for r in judge_results]
sample_df['judge_explanation'] = [r.get('explanation', '') for r in judge_results]

print('\n=== LLM-as-Judge Results ===')
for col in ['faithful', 'key_point', 'misleading']:
    counts = sample_df[col].value_counts()
    print(f'{col}: {dict(counts)}')


## Side-by-Side Comparison


In [ ]:
for idx, row in sample_df.iterrows():
    print(f'--- Article {idx} ---')
    print(f'Reference:  {row["reference_summary"]}')
    print(f'LLM:        {row["llm_summary"]}')
    print(f'Faithful: {row["faithful"]}  |  Key point: {row["key_point"]}  |  Misleading: {row["misleading"]}')
    if row['judge_explanation']:
        print(f'Judge note: {row["judge_explanation"]}')
    print()


## Discussion Questions

1. **Was key information lost?** Compare the reference and LLM summaries.
   Where does the LLM lose important nuance?

2. **Was anything hallucinated?** Did the LLM add facts not in the article?
   This is the #1 summarization failure mode.

3. **Can you trust the LLM judge?** The judge is also an LLM — it can make mistakes too.
   This is a real tension in production evaluation systems.

## Export


In [ ]:
export_df = sample_df[['reference_summary', 'llm_summary', 'llm_word_count', 'faithful', 'key_point', 'misleading', 'judge_explanation']].copy()
export_df['student_notes'] = ''
export_path = 'summarization_results.xlsx'
export_df.to_excel(export_path, index=False)
print(f'Saved {export_path}')

try:
    from google.colab import files
    files.download(export_path)
except Exception:
    print('Download the file from the notebook working directory.')
